# Mudra — Train All 262 Phrases on Free Google Colab GPU

This notebook trains the **Bidirectional GRU with Temporal Average + Max Pooling** on all **262 classes** of the INCLUDE dataset.
Using a free Colab T4 GPU, training takes under **2 minutes** and exports the optimized ONNX model.

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Install dependencies
!pip install onnx onnxscript onnxruntime numpy

In [ ]:
# 3. Upload include_landmarks.npz (or download from Google Drive)
from google.colab import files
print("Please upload your 'include_landmarks.npz' file (from mudra/training/data/landmarks/):")
uploaded = files.upload()
npz_path = list(uploaded.keys())[0]

In [ ]:
# 4. Define Architecture
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import json
import os

class LandmarkNormalization(nn.Module):
    def forward(self, landmarks, hand_mask):
        B = landmarks.size(0)
        m = hand_mask.unsqueeze(2).unsqueeze(3)
        L = landmarks * m
        sum_xy = L[..., :2].sum(dim=(1, 2))
        num_points = (21.0 * hand_mask.sum(dim=1, keepdim=True)).clamp(min=1e-6)
        center_xy = sum_xy / num_points
        zeros_z = torch.zeros_like(center_xy[:, :1])
        center_3d = torch.cat([center_xy, zeros_z], dim=-1).unsqueeze(1).unsqueeze(2)
        xy_diff_global = (L[..., :2] - center_3d[..., :2]) * m[..., :2]
        dist_global = torch.norm(xy_diff_global, dim=-1)
        scale_global = dist_global.flatten(1).max(dim=1)[0].clamp(min=1e-6).view(B, 1, 1, 1)
        feat_global = ((L - center_3d) / scale_global) * m
        wrist_xy = L[:, :, 0:1, :2]
        wrist_3d = torch.cat([wrist_xy, torch.zeros_like(wrist_xy[..., :1])], dim=-1)
        xy_diff_local = (L[..., :2] - wrist_xy) * m[..., :2]
        dist_local = torch.norm(xy_diff_local, dim=-1)
        scale_local = dist_local.max(dim=2, keepdim=True)[0].unsqueeze(-1).clamp(min=1e-6)
        feat_local = ((L - wrist_3d) / scale_local) * m
        feat_global_flat = feat_global.view(B, -1)
        feat_local_flat = feat_local.view(B, -1)
        return torch.cat([feat_global_flat, feat_local_flat, hand_mask], dim=1)

class FullINCLUDEBiGRU(nn.Module):
    def __init__(self, num_classes, hidden_size=256):
        super().__init__()
        self.norm = LandmarkNormalization()
        self.gru = nn.GRU(
            input_size=254,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.35
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 4, 384),
            nn.BatchNorm1d(384),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(384, num_classes)
        )

    def forward(self, lms_seq, mask_seq):
        B, T = lms_seq.shape[:2]
        lms_flat = lms_seq.reshape(B * T, 2, 21, 3)
        mask_flat = mask_seq.reshape(B * T, 2)
        features = self.norm(lms_flat, mask_flat)
        features_seq = features.reshape(B, T, -1)
        gru_out, _ = self.gru(features_seq)
        mean_pool = torch.mean(gru_out, dim=1)
        max_pool = torch.max(gru_out, dim=1)[0]
        pooled = torch.cat([mean_pool, max_pool], dim=-1)
        return self.classifier(pooled)

class ExportWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.softmax = nn.Softmax(dim=1)
    def forward(self, lms_seq, mask_seq):
        logits = self.model(lms_seq, mask_seq)
        return self.softmax(logits)

print("Model architecture defined successfully.")

In [ ]:
# 5. Train with 80/20 Validation
d = np.load(npz_path)
X, M, Y = d["landmarks"], d["hand_mask"], d["labels"]
raw_names = d["class_names"].tolist()
clean_names = [c.split(". ")[-1].strip() if ". " in c else c for c in raw_names]
num_classes = len(clean_names)

np.random.seed(42)
torch.manual_seed(42)
indices = np.arange(len(X))
np.random.shuffle(indices)
split = int(0.8 * len(X))
tr_idx, val_idx = indices[:split], indices[split:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on {device} across {num_classes} classes...")

model = FullINCLUDEBiGRU(num_classes, hidden_size=256).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

tr_loader = DataLoader(TensorDataset(torch.tensor(X[tr_idx]), torch.tensor(M[tr_idx]), torch.tensor(Y[tr_idx], dtype=torch.long)), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X[val_idx]), torch.tensor(M[val_idx]), torch.tensor(Y[val_idx], dtype=torch.long)), batch_size=64, shuffle=False)

epochs = 35
best_val_acc = 0.0
best_state = None

for epoch in range(1, epochs + 1):
    model.train()
    tr_loss, tr_correct, tr_total = 0, 0, 0
    for lms, masks, labels in tr_loader:
        lms, masks, labels = lms.to(device), masks.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(lms, masks)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * lms.size(0)
        preds = torch.argmax(out, dim=1)
        tr_correct += (preds == labels).sum().item()
        tr_total += labels.size(0)

    tr_acc = tr_correct / tr_total
    tr_loss /= tr_total

    model.eval()
    val_loss, val_correct, val_total, top5_correct = 0, 0, 0, 0
    with torch.no_grad():
        for lms, masks, labels in val_loader:
            lms, masks, labels = lms.to(device), masks.to(device), labels.to(device)
            out = model(lms, masks)
            loss = criterion(out, labels)
            val_loss += loss.item() * lms.size(0)
            preds = torch.argmax(out, dim=1)
            val_correct += (preds == labels).sum().item()
            _, top5 = torch.topk(out, 5, dim=1)
            top5_correct += torch.sum(top5 == labels.unsqueeze(1)).item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    val_top5 = top5_correct / val_total
    val_loss /= val_total
    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = model.state_dict()

    if epoch % 5 == 0 or epoch == epochs:
        print(f"Epoch {epoch:02d}/{epochs} | Train Acc: {tr_acc*100:.1f}%, Loss: {tr_loss:.3f} | Val Top-1: {val_acc*100:.1f}%, Val Top-5: {val_top5*100:.1f}%, Loss: {val_loss:.3f} | Best Val: {best_val_acc*100:.1f}%")

print(f"\nPeak Validation Top-1 Accuracy: {best_val_acc*100:.2f}%")

In [ ]:
# 6. Export to ONNX and Download
if best_state is not None:
    model.load_state_dict(best_state)

model.cpu()
model.eval()
wrapper = ExportWrapper(model)
onnx_filename = "mudra_phrases.onnx"

dummy_lms = torch.randn(1, 30, 2, 21, 3)
dummy_mask = torch.ones(1, 30, 2)

torch.onnx.export(
    wrapper,
    (dummy_lms, dummy_mask),
    onnx_filename,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["landmarks", "hand_mask"],
    output_names=["probabilities"],
    dynamic_axes=None
)

meta = {
    "class_names": clean_names,
    "class_count": len(clean_names),
    "type": "sequence",
    "validation_accuracy": round(best_val_acc * 100, 2)
}
with open("mudra_phrases_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("ONNX export complete. Downloading files...")
files.download(onnx_filename)
files.download("mudra_phrases_meta.json")